# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pranavtilekar9999/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Find data/raw/content_refresh_anonymized.csv and run from the repo root.")
df = pd.read_csv(data_path)
required = {"content_id", "days_since_last_update", "impressions_90d"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Required columns missing: {sorted(missing)}")
print(f"Loaded {data_path}; rows={len(df):,}, columns={len(df.columns)}")
print("Missing rule inputs:")
print(df[["days_since_last_update", "impressions_90d"]].isna().sum())
print("Duplicate content IDs:", int(df["content_id"].duplicated().sum()))

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
df["days_since_last_update"] = pd.to_numeric(df["days_since_last_update"], errors="coerce")
df["impressions_90d"] = pd.to_numeric(df["impressions_90d"], errors="coerce")
stale = df["days_since_last_update"].ge(180).fillna(False)
visible = df["impressions_90d"].ge(300).fillna(False)
qualifies = stale & visible

queue = pd.DataFrame({
    "content_id": df["content_id"].astype(str),
    "days_since_last_update": df["days_since_last_update"],
    "impressions_90d": df["impressions_90d"],
    "score": np.where(qualifies, df["impressions_90d"].fillna(0), 0.0),
    "reason_code": np.where(qualifies, "stale_but_visible", "below_rule_threshold"),
    "action": np.where(qualifies, "refresh_review", "monitor_or_lower_priority"),
})
queue = queue.sort_values(
    ["score", "impressions_90d", "content_id"],
    ascending=[False, False, True], na_position="last", kind="mergesort"
).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)
n = int(qualifies.sum())
print(f"Rows ranked: {len(queue):,}")
print(f"Rows meeting refresh-review rule: {n:,} ({n / len(queue):.1%})")
print(f"CSV written to: {output_path}")
display(queue.head(10))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue.head(20).copy()

def confidence(r):
    if r["reason_code"] != "stale_but_visible":
        return "Low priority under this rule; both thresholds are not met."
    if r["days_since_last_update"] >= 365 and r["impressions_90d"] >= 3000:
        return "Moderate confidence in review priority: clearly stale and visible."
    return "Moderate-to-low confidence: thresholds pass, but refresh value is unproven."

def caveat(r):
    if r["reason_code"] != "stale_but_visible":
        return "Could be wrongly deprioritized if it has high business value or an urgent issue."
    notes = []
    if r["impressions_90d"] < 1000:
        notes.append("visibility is modest")
    if r["days_since_last_update"] < 365:
        notes.append("staleness is near the cutoff")
    notes.append("intent, content quality, and business value are not represented")
    return "Could be a weak pick if " + "; ".join(notes) + "."

top20["confidence_note"] = top20.apply(confidence, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(caveat, axis=1)
for _, r in top20.iterrows():
    print(f"Rank {int(r['rank'])} | ID {r['content_id']} | Action: {r['action']} | "
          f"Reason: {r['reason_code']} | Confidence: {r['confidence_note']} | "
          f"What could make it wrong: {r['what_would_make_it_wrong']}")
display(top20[["rank","content_id","days_since_last_update","impressions_90d",
               "score","action","reason_code","confidence_note","what_would_make_it_wron

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
prioritized = queue.loc[queue["reason_code"].eq("stale_but_visible")].copy()
weak = prioritized.loc[
    prioritized["impressions_90d"].lt(1000)
    | prioritized["days_since_last_update"].lt(210)
].head(10)
print(f"Prioritized rows: {len(prioritized):,}")
if weak.empty:
    print("No rows met this simple weak-pick screen.")
else:
    print("Potential weak picks (not automatically errors):")
    display(weak[["rank","content_id","days_since_last_update","impressions_90d",
                  "score","reason_code","action"]])
    print("Check intent, business value, content accuracy, and whether a refresh is appropriate.")

rule_inputs = {"days_since_last_update", "impressions_90d"}
forbidden = ("is_declining_label","trend_direction","trend_pct","flag",
             "product_score","future","next_30d","next_90d")
hits = [c for c in rule_inputs if any(term in c.lower() for term in forbidden)]
assert not hits, f"Potential leakage inputs: {hits}"
assert set(queue["reason_code"].unique()) <= {"stale_but_visible","below_rule_threshold"}
assert len(queue) == len(df) and queue["rank"].is_unique
print("Leakage check passed: only days_since_last_update and impressions_90d drive the score.")
print("Keep work/outputs/baseline_action_score.csv out of git.")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.